# Pulsatile MHD Casson Flow — POD-ROM + PINN-ROM

Jupyter notebook to run the full experiment.

**Steps:**
1. Install dependencies (run once)
2. Run the experiment
3. View tables and figures

Reference paper: *Pulsatile MHD flow of a Casson fluid through a porous bifurcated arterial stenosis under periodic body acceleration* (Ponalagusamy & Priyadharshini, AMC 2018)

## 1. Install dependencies (run once)

In [ ]:
%pip install numpy scipy matplotlib pandas torch

## 2. Setup paths and imports

In [ ]:
import sys
import os

# Set project root - this notebook should be at the project root
PROJECT_ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import project modules
from src.casson_fom import FlowParams, FOMSolver, generate_snapshots
from src.pod_rom import PODBasis, GalerkinROM, build_pod_rom
from src.pinn_rom import PINNROM, PINNROMTrainer, build_pinn_rom
from src.postproc import save_all_results
print('Imports OK')

## 3. Define parameter samples

In [ ]:
# Quick test set (4 samples) - takes ~10 seconds
param_samples = [
    {'M': 1.0, 'Da': 0.5, 'theta': 0.05, 'delta': 0.2, 'beta_half': np.pi/6},
    {'M': 2.0, 'Da': 0.5, 'theta': 0.1,  'delta': 0.3, 'beta_half': np.pi/6},
    {'M': 3.0, 'Da': 0.3, 'theta': 0.15, 'delta': 0.4, 'beta_half': np.pi/4},
    {'M': 2.0, 'Da': 1.0, 'theta': 0.1,  'delta': 0.3, 'beta_half': np.pi/5},
]
print(f'{len(param_samples)} parameter samples defined')

## 4. Generate FOM snapshots (Full Order Model)

In [ ]:
fom_results = generate_snapshots(param_samples, Nr=80, Nt=150, n_cycles=2)
print(f"\nSnapshot matrix shape: ({fom_results['r'].shape[0]}, {len(fom_results['t'])})")

## 5. Build POD basis and Galerkin ROM

In [ ]:
representative_params = {
    'M': 2.0, 'Da': 0.5, 'theta': 0.1, 'A0': 1.0,
    'e': 0.5, 'a_b': 0.5, 'omega_b': 1.0, 'phi': 0.0,
    'alpha_sq': 1.0, 'eps': 1e-3,
}
pod_basis, galerkin_rom = build_pod_rom(fom_results, representative_params, energy_threshold=0.999)
print(f'POD modes retained: {pod_basis.k}')
print(f'Energy captured:    {pod_basis.energy_captured:.4f}')

## 6. Train PINN-ROM

In [ ]:
pinn_trainer = build_pinn_rom(
    pod_basis, galerkin_rom, fom_results, param_samples,
    n_epochs=1500, lr=1e-3,
)

## 7. Generate tables and figures

In [ ]:
save_all_results(fom_results, param_samples, pod_basis, pinn_trainer, output_dir='results')

## 8. View tables

In [ ]:
print('=== Wall Shear Stress ===')
display(pd.read_csv('results/tables/wss.csv'))

In [ ]:
print('=== Flow Resistance ===')
display(pd.read_csv('results/tables/resistance.csv'))

In [ ]:
print('=== Plug Core Radius ===')
display(pd.read_csv('results/tables/plug_core.csv'))

## 9. View figures

In [ ]:
from IPython.display import Image, display
import os

fig_dir = 'results/figs'
for fig_name in sorted(os.listdir(fig_dir)):
    if fig_name.endswith('.png'):
        print(f'\n=== {fig_name} ===')
        display(Image(os.path.join(fig_dir, fig_name)))

## 10. Custom prediction at unseen parameter values

In [ ]:
# Predict velocity profile at NEW parameter combination not used in training
new_params = {'M': 2.5, 'Da': 0.7, 'theta': 0.08, 'delta': 0.25, 'beta_half': np.pi/6}
t_query = fom_results['t']
U_pinn = pinn_trainer.predict_velocity(t_query, new_params)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(fom_results['r'], U_pinn[:, len(t_query)//2], 'r--', label='t=T/2', linewidth=2)
ax.plot(fom_results['r'], U_pinn[:, -1], 'b-', label='t=T', linewidth=2)
ax.set_xlabel('r')
ax.set_ylabel('u(r,t)')
ax.set_title(f"PINN-ROM prediction at unseen params: {new_params}")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()